In [1]:
!pip install -q "psycopg[binary]" pgvector pypdf tiktoken requests numpy python-dotenv

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import os
from urllib.parse import urlparse

try:
    from dotenv import load_dotenv
    _env_file_found = bool(load_dotenv())
except ImportError:
    _env_file_found = False

DB_URI = os.environ.get("DATABASE_URL2")
HF_TOKEN = os.environ.get("HF_TOKEN")

if not DB_URI or not HF_TOKEN:
    raise ValueError(
        "❌ DATABASE_URL2 یا HF_TOKEN پیدا نشد.\n"
        "   راه ۱: کنار این نوت‌بوک فایل .env بسازید (الگو: env.example) و سلول را دوباره اجرا کنید.\n"
        "   راه ۲: متغیرها را قبل از اجرای jupyter در ترمینال ست کنید (export / set).\n"
        "   توکن HF را از huggingface.co/settings/tokens بگیرید (دسترسی: Make calls to Inference Providers).\n"
    )

# نمایش امن: فقط هاستِ دیتابیس و انتهای توکن — بدون پسورد و توکن کامل
print("✅ Secrets loaded"
      + (" از فایل .env" if _env_file_found else " از متغیرهای محیطی سیستم (فایل .env پیدا نشد)"))
print(f"   Supabase host : {urlparse(DB_URI).hostname}")
print(f"   HF token      : ****{HF_TOKEN[-4:]}")

✅ Secrets loaded از فایل .env
   Supabase host : aws-1-eu-west-1.pooler.supabase.com
   HF token      : ****LlcA


## 🤗 Embedding با Hugging Face Inference API — نسخه‌ی اجرای محلی (لپ‌تاپ)

مدل `BAAI/bge-large-en-v1.5` **بدون دانلود** از طریق **Hugging Face Inference API** سرو می‌شود:

- بعدِ بردار همان **1024** می‌ماند → ستون `vector(1024)` جدول Supabase و همه‌ی داده‌های قبلی بدون تغییر قابل استفاده‌اند.
- مدل دقیقاً همان است → بردارهای جدید با بردارهای قبلیِ محاسبه‌شده سازگاری دارند؛ لازم نیست دیتابیس از صفر ری‌ایندکس شود.
- چون محاسبه سمت سرور انجام می‌شود، روی لپ‌تاپ **نه GPU لازم است و نه RAM زیاد**.
- این مدل در HF روی دو پروایدر فعال (live) است: `hf-inference` و `deepinfra`. این نوت‌بوک از `hf-inference` استفاده می‌کند.

**راه‌اندازی (سه قدم):**

1. توکن را از [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) بسازید و هنگام ساخت، دسترسی **Make calls to Inference Providers** را به آن بدهید.
2. فایل `env.example` را به `.env` تغییر نام دهید، کنار همین نوت‌بوک بگذارید و دو مقدار را پر کنید: `DATABASE_URL2` و `HF_TOKEN`
3. مسیر فایل PDF را در سلول بعدی ست کنید، سپس سلول‌ها را به ترتیب اجرا کنید.

**هزینه:** HF به اکانت‌های معمولی اعتبار ماهانه رایگان نمی‌دهد؛ یا credit شارژ کنید یا PRO بگیرید (شامل ۲ دلار اعتبار ماهانه). هزینه‌ی embed کردن کل این کتاب چند سنت است.

**جایگزین‌ها** (برای عوض کردن، فقط `EMBEDDING_API_URL` در سلول اصلی را عوض کنید):

| پروایدر | مدل | ابعاد | توضیح |
|---|---|---|---|
| HF — hf-inference | `BAAI/bge-large-en-v1.5` | 1024 | همین نوت‌بوک ✅ |
| DeepInfra (مستقیم) | `BAAI/bge-large-en-v1.5` | 1024 | API سازگار با OpenAI — نسخه‌ی `chunk_api_local.ipynb` |
| OpenAI | `text-embedding-3-large` + `dimensions=1024` | 1024 | مدل متفاوت است → باید **کل دیتابیس** ری‌ایندکس شود و در query هم همین مدل استفاده شود |
| OpenAI | `text-embedding-3-small` | 1536 | مدل و ابعاد متفاوت → تغییر schema لازم است |

> ⚠️ قاعده‌ی کلی: هر وقت **مدل embedding عوض شود**، بردارهای قدیمی با بردارهای جدید قابل مقایسه نیستند؛ یا کل جدول را drop و دوباره ingest کنید، یا همان مدل قبلی را نگه دارید.

In [3]:
import psycopg
from pgvector.psycopg import register_vector

with psycopg.connect(DB_URI) as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        conn.commit()

    register_vector(conn)

    with conn.cursor() as cur:
        create_table_query = """
        DROP TABLE IF EXISTS book_chunks;
        CREATE TABLE IF NOT EXISTS book_chunks (
            id SERIAL PRIMARY KEY,
            chunk_id VARCHAR(100) UNIQUE NOT NULL,
            source_title VARCHAR(255) NOT NULL,
            page_number INT NOT NULL,
            content TEXT NOT NULL,
            token_count INT NOT NULL,
            embedding vector(1024)
        );

        CREATE INDEX IF NOT EXISTS book_chunks_embedding_idx
        ON book_chunks USING hnsw (embedding vector_cosine_ops);
        """
        cur.execute(create_table_query)
        conn.commit()

print("done")


done


In [4]:
# ---------- مسیر PDF روی لپ‌تاپ ----------
import os

# ویندوز: از رشته‌ی خام استفاده کنید، مثلاً:  PDF_PATH = r"C:\Users\You\Desktop\book.pdf"
PDF_PATH = "../../../AI Engineering.pdf"  # <-- مسیر فایل PDF را اینجا ست کنید

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"❌ فایل PDF پیدا نشد: {PDF_PATH}\n"
        f"   پوشه‌ی جاری: {os.getcwd()}\n"
        "   مسیر PDF_PATH را در همین سلول اصلاح کنید (این مقدار در سلول‌های بعدی هم استفاده می‌شود)."
    )
print(f"✅ PDF found: {os.path.abspath(PDF_PATH)}")

✅ PDF found: /home/itsmohsen/Desktop/Bachelor_Project/Bachelor-s-Project-I-LMS/AI Engineering.pdf


In [7]:
import pypdf

PDF_FILE_PATH = PDF_PATH  # از سلول قبل می‌آید — دیگر نیازی به Google Drive نیست

reader = pypdf.PdfReader(PDF_FILE_PATH)
print(f"book pages: {len(reader.pages)}")

sample_page = reader.pages[5]
print("sample of extracted sentece")
print(sample_page.extract_text()[:500])

book pages: 535
sample of extracted sentece
978-1-098-16630-4
[LSI]
AI Engineering
by Chip Huyen
Copyright © 2025 Developer Experience Advisory LLC. All rights reserved.
Printed in the United States of America.
Published by O’Reilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.
O’Reilly books may be purchased for educational, business, or sales promotional use. Online editions are
also available for most titles ( http://oreilly.com). For more information, contact our corporate/institu‐
tional sales department: 800-998


In [8]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

In [10]:
import os
import time
import requests
import numpy as np
import tiktoken
from pypdf import PdfReader
import psycopg
from pgvector.psycopg import register_vector

# ================= Configuration =================
# DB_URI و HF_TOKEN در سلول اول از .env خوانده شده‌اند.
# PDF_PATH در سلول مسیر PDF ست شده است.
SOURCE_TITLE = "Computer Networking: A Top-Down Approach, 8th Edition"

# --- Embedding API (Hugging Face Inference — hf-inference provider) ---
EMBEDDING_API_URL = "https://router.huggingface.co/hf-inference/models/BAAI/bge-large-en-v1.5/pipeline/feature-extraction"
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"  # 1024 dimensions
API_BATCH_SIZE = 32      # تعداد متن در هر درخواست HTTP
MAX_RETRIES = 5          # تلاش مجدد روی خطاهای 429/5xx
REQUEST_TIMEOUT = 120    # ثانیه

CHUNK_SIZE = 500       # Target tokens per chunk
CHUNK_OVERLAP = 50     # Overlap tokens between chunks
BATCH_SIZE = 50        # Database batch insert size

# For full book: START_PAGE = 0, END_PAGE = None
START_PAGE = 0
END_PAGE = 10       # Set to None to automatically process all pages
# =================================================


def get_token_encoder():
    return tiktoken.get_encoding("cl100k_base")


def extract_chunks_from_pdf(pdf_path, start_page=0, end_page=None):
    print(f"📖 Loading PDF from: {pdf_path}")
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    print(f"📑 Total pages detected: {total_pages}")

    if end_page is None or end_page > total_pages:
        end_page = total_pages

    tokenizer = get_token_encoder()
    chunks = []
    chunk_counter = 1

    print(f"⚙️ Chunking pages {start_page} to {end_page}...")

    for page_idx in range(start_page, end_page):
        page = reader.pages[page_idx]
        text = page.extract_text() or ""

        # Clean basic whitespace
        text = " ".join(text.split())
        if not text:
            continue

        tokens = tokenizer.encode(text)
        page_num = page_idx + 1  # 1-based page number for human readability

        # Sliding window chunking
        for i in range(0, len(tokens), CHUNK_SIZE - CHUNK_OVERLAP):
            chunk_tokens = tokens[i:i + CHUNK_SIZE]
            chunk_text = tokenizer.decode(chunk_tokens)

            chunk_id = f"chunk_p{page_num}_{chunk_counter}"
            chunk_counter += 1

            chunks.append({
                "chunk_id": chunk_id,
                "page_number": page_num,
                "content": chunk_text,
                "token_count": len(chunk_tokens)
            })

    print(f"✅ Generated {len(chunks)} chunks in total.")
    return chunks


def _to_sentence_matrix(raw, n_expected):
    """خروجی API را به ماتریس (n, dim) بردارِ جمله تبدیل می‌کند."""
    if isinstance(raw, dict) and "embeddings" in raw:
        raw = raw["embeddings"]  # بعضی پروایدرها دیکشنری برمی‌گردانند
    arr = np.asarray(raw, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    if arr.ndim == 3:
        raise RuntimeError(
            "API بردار توکن‌به‌توکن برگرداند (به‌جای بردار جمله). "
            "ادامه دادن یعنی بردارهای ناسازگار با دیتابیس فعلی — متوقف شد."
        )
    if arr.ndim != 2 or arr.shape[0] != n_expected:
        got = np.asarray(raw).shape
        raise RuntimeError(f"Unexpected embedding response shape {got} (expected {n_expected} rows)")
    return arr


def embed_texts(texts, api_key, batch_size=API_BATCH_SIZE):
    """
    Embedding دسته‌ای با HF Inference API (به‌جای SentenceTransformer محلی).
    خروجی: آرایه numpy به شکل (len(texts), 1024) — نرمال‌شده L2 (معادل normalize_embeddings=True)

    جایگزین با SDK رسمی (اختیاری):
        from huggingface_hub import InferenceClient
        client = InferenceClient(api_key=HF_TOKEN)
        emb = client.feature_extraction(texts, model=EMBEDDING_MODEL_NAME)
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    all_embeddings = [None] * len(texts)
    num_batches = (len(texts) + batch_size - 1) // batch_size

    print(f"🚀 Calculating embeddings via Hugging Face API: {EMBEDDING_MODEL_NAME}")
    print(f"   {len(texts)} texts → {num_batches} HTTP requests (batch={batch_size})")

    for b in range(num_batches):
        start = b * batch_size
        batch_texts = texts[start:start + batch_size]

        payload = {
            "inputs": batch_texts,
            "truncate": True,  # معادل truncate خودکار sentence-transformers در 512 توکن
        }

        # retry با backoff نمایی روی خطاهای موقت (rate limit / server error)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = requests.post(
                    EMBEDDING_API_URL,
                    headers=headers,
                    json=payload,
                    timeout=REQUEST_TIMEOUT,
                )
                if resp.status_code == 200:
                    break
                # خطای قابل تلاش مجدد: 429 یا 5xx
                if resp.status_code in (429, 500, 502, 503, 504):
                    wait = 2 ** attempt
                    print(f"   ⏳ HTTP {resp.status_code}, retry {attempt}/{MAX_RETRIES} in {wait}s")
                    time.sleep(wait)
                    continue
                # خطای غیرقابل بازیابی (مثلاً 401 توکن اشتباه یا 404 مدل سرو نمی‌شود)
                raise RuntimeError(
                    f"HF Inference API error HTTP {resp.status_code}: {resp.text[:300]}"
                )
            except requests.RequestException as e:
                wait = 2 ** attempt
                print(f"   ⏳ Network error ({e}), retry {attempt}/{MAX_RETRIES} in {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError(f"HF Inference API failed after {MAX_RETRIES} retries (batch {b + 1}).")

        batch_emb = _to_sentence_matrix(resp.json(), len(batch_texts))

        # L2 normalize — معادل normalize_embeddings=True در SentenceTransformer
        norms = np.linalg.norm(batch_emb, axis=1, keepdims=True)
        if (norms == 0).any():
            raise RuntimeError("Zero-norm embedding returned — check input texts.")
        batch_emb = batch_emb / norms

        for j, emb in enumerate(batch_emb):
            all_embeddings[start + j] = emb

        if (b + 1) % 5 == 0 or (b + 1) == num_batches:
            done = min(start + batch_size, len(texts))
            print(f"   🧠 Embedded {done}/{len(texts)} texts...")

    embeddings = np.asarray(all_embeddings, dtype=np.float32)
    if np.isnan(embeddings).any() or embeddings.shape[1] != 1024:
        raise RuntimeError(f"Unexpected embedding shape/dtype: {embeddings.shape}")
    return embeddings


def main():
    # 1. Chunk the PDF
    chunks = extract_chunks_from_pdf(PDF_PATH, START_PAGE, END_PAGE)
    if not chunks:
        print("⚠️ No chunks found. Please verify PDF path and page range.")
        return

    # 2. HF token (از سلول اول — .env) — بدون دانلود هیچ مدلی
    api_key = HF_TOKEN
    if not api_key:
        raise ValueError(
            "❌ HF_TOKEN ست نشده. فایل .env را چک کنید و سلول اول را دوباره اجرا کنید."
        )

    # 3. Generate Embeddings via API
    texts_to_embed = [chunk["content"] for chunk in chunks]
    embeddings = embed_texts(texts_to_embed, api_key)

    # 4. Connect to Supabase & Batch Insert
    print("🔌 Connecting to Supabase database...")
    insert_query = """
        INSERT INTO book_chunks (
            chunk_id,
            source_title,
            page_number,
            content,
            token_count,
            embedding
        ) VALUES (%s, %s, %s, %s, %s, %s);
    """

    with psycopg.connect(DB_URI) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            print("💾 Inserting chunks into Supabase table 'book_chunks'...")

            records_batch = []
            for idx, (chunk, emb) in enumerate(zip(chunks, embeddings)):
                records_batch.append((
                    chunk["chunk_id"],
                    SOURCE_TITLE,
                    chunk["page_number"],
                    chunk["content"],
                    chunk["token_count"],
                    emb
                ))

                # Commit in batches to optimize network I/O
                if len(records_batch) >= BATCH_SIZE:
                    cur.executemany(insert_query, records_batch)
                    conn.commit()
                    records_batch.clear()
                    print(f"📦 Committed {idx + 1}/{len(chunks)} chunks...")

            # Insert any remaining records
            if records_batch:
                cur.executemany(insert_query, records_batch)
                conn.commit()

    print("🎉 Ingestion complete! All chunks and vector embeddings successfully saved to Supabase.")


if __name__ == "__main__":
    main()

📖 Loading PDF from: ../../../AI Engineering.pdf
📑 Total pages detected: 535
⚙️ Chunking pages 0 to 10...


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/WGDTRK+Helvetica', '/Encoding': IndirectObject(563, 0, 126898012115248), '/FirstChar': 23, '/FontDescriptor': IndirectObject(564, 0, 126898012115248), '/LastChar': 31, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [556, 556, 556, 556, 556, 556, 556, 556, 556]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


✅ Generated 10 chunks in total.
🚀 Calculating embeddings via Hugging Face API: BAAI/bge-large-en-v1.5
   10 texts → 1 HTTP requests (batch=32)


RuntimeError: HF Inference API error HTTP 402: {"error":"Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits."}